### 🖼️ Dataset Thumbnails (minpeter___fineweb-2-edu-korean-raw)

![Thumbnail](../thumbnails/minpeter___fineweb-2-edu-korean-raw_01.png)

In [ ]:
import random
from datasets import load_dataset, Dataset, IterableDataset
import time
from typing import List, Dict, Any

# =============================================================================
# ✨ 데이터셋 소개: minpeter/fineweb-2-edu-korean-raw
# 📜 한글 제목: 한국 교육 웹 데이터셋 (Korean Educational Web Dataset)
# 💡 의미 및 설명: 이 데이터셋은 한국어(ko)로 작성된 다양한 웹 페이지에서 수집된 대규모 교육 콘텐츠를 모아놓은 것입니다.
# 이 데이터를 사용하면 '어떤 주제의 웹 콘텐츠가 학습에 유용할까?', '어떤 출처의 자료가 신뢰도가 높을까?' 등을 탐색해 볼 수 있습니다.
# 우리는 이 데이터를 활용하여 '최고의 학습 콘텐츠'를 찾는 초보 탐정 놀이를 해볼 거예요!
# =============================================================================

# --- 상수 설정 ---
DATASET_NAME = "minpeter/fineweb-2-edu-korean-raw"
SAMPLE_COUNT = 50  # 탐색을 위해 상위 50개 샘플만 사용합니다. 너무 많으면 느려져요!

print("✨ 학생 여러분, 파이썬 코딩 탐험을 시작해 볼까요? 데이터 분석을 통해 숨겨진 보물을 찾아봅시다!")
print("==============================================================================\n")


def load_data_robustly(dataset_id: str, split: str, sample_count: int) -> List[Dict[str, Any]]:
    """
    스트리밍 여부를 확인하며 데이터셋을 로드하고, 샘플 데이터를 리스트로 반환합니다.
    (가장 중요하고 까다로운 부분입니다! 안정적으로 코드가 돌아가게 해줍니다.)
    """
    print(f"🔗 데이터를 로드합니다... ({dataset_id}, split='{split}')")
    
    # 1. 스트리밍 가능 여부 확인
    if hasattr(load_dataset(dataset_id, split=split, streaming=True), "take"):
        print("✅ 스트리밍 모드를 시도합니다. (데이터가 커도 메모리 부담이 적어요!)")
        try:
            # 스트리밍 모드로 로드 시도
            dataset = load_dataset(dataset_id, split=split, streaming=True)
            
            # 스트리밍 데이터셋의 특성상, 전체 크기 확인(len())이 불가능합니다.
            # 따라서 take()을 사용해 메모리 효율적으로 샘플을 추출해야 합니다.
            print(f"💡 메모리 효율적인 스트리밍 탐색 모드 사용.")
            
            # take()를 사용해 상위 K개의 이터레이터(iterator)를 만듭니다.
            sample_iterator = dataset.take(sample_count)
            
            # 이터레이터에서 리스트로 변환 (가장 안전한 방법!)
            sample_data_list = list(sample_iterator)
            return sample_data_list

        except Exception as e:
            print(f"⚠️ 스트리밍 모드 로드 중 오류가 발생했어요: {e}")
            print("➡️ 일반(Non-Streaming) 모드로 전환하여 작은 샘플만 로드하겠습니다.")
            
            # 스트리밍 실패 시, 일반 모드로 적은 수만 로드하여 중단 방지
            try:
                dataset = load_dataset(dataset_id, split=split, streaming=False)
                # 샘플만 로드 (메모리 절약)
                return list(dataset.select(range(min(sample_count, len(dataset)))))
            except Exception as e_fail:
                print(f"❌ 데이터셋 로드에 실패했습니다: {e_fail}")
                return []
    else:
        # 스트리밍이 불가능한 경우 (일반 Dataset 객체)
        print("🔄 스트리밍이 불가능한 데이터셋입니다. 일반 로딩을 진행합니다.")
        dataset = load_dataset(dataset_id, split=split)
        # select(index_list)를 사용하여 상위 K개만 가져옵니다.
        return list(dataset.select(range(min(sample_count, len(dataset)))))


# -----------------------------------------------------------------------------
# 🚀 메인 실습 함수: 데이터 탐험가 미션!
# -----------------------------------------------------------------------------
def run_data_explorer(data_list: List[Dict[str, Any]]):
    """
    로드된 데이터 리스트를 활용하여 재미있고 창의적인 분석을 진행합니다.
    """
    if not data_list:
        print("\n\n😭 분석할 데이터가 준비되지 않았습니다. 스크립트를 종료합니다.")
        return

    print(f"\n\n🎨 ✨ [미션 성공!] 상위 {len(data_list)}개의 교육 콘텐츠 샘플을 확보했습니다.")
    print("이제 이 데이터 속에서 '꿀팁'을 찾아보겠습니다!")

    # 1. 기본 통계 분석 (어떤 종류의 콘텐츠가 많은가?)
    
    # 우리가 알고 싶은 것: 가장 많이 등장하는 언어와 점수 평균
    language_counts = {}
    total_scores = 0.0
    
    print("\n======== 🔍 [Step 1] 기초 통계 분석: 콘텐츠의 언어와 품질 파악 ========")
    print("데이터 전체를 관찰하며 '언어 분포'와 '평균 학습 점수'를 계산해 봅시다.")
    
    for i, sample in enumerate(data_list):
        # 필요한 정보만 추출 (언어, 점수)
        language = sample.get('language', 'N/A')
        score = sample.get('score', 0.0)
        
        # 언어 빈도수 계산
        language_counts[language] = language_counts.get(language, 0) + 1
        total_scores += score

    avg_score = total_scores / len(data_list) if data_list else 0.0
    
    print(f"\n[📈 통계 결과 요약]")
    print(f"총 샘플 수: {len(data_list)}개")
    print(f"✔️ 가장 많은 콘텐츠의 언어: {max(language_counts, key=language_counts.get)} ({max(language_counts, key=language_counts.get)} 언어로 {language_counts[max(language_counts, key=language_counts.get)]}개)")
    print(f"✨ 평균 학습 점수 (Score): {avg_score:.4f}점")


    # 2. 특성 필터링 및 탐색 (조건에 맞는 콘텐츠만 뽑아내기)
    
    print("\n======== 🔎 [Step 2] 필터링 미션: 고득점의 교육 콘텐츠만 찾기 ========")
    # 목표: 점수(score)가 높고, 특정 키워드를 포함하는 콘텐츠만 걸러내기
    
    high_score_samples = []
    target_keywords = ['교육', '학습', '과학', '역사'] # 예시 키워드
    
    for i, sample in enumerate(data_list):
        score = sample.get('score', 0.0)
        text = sample.get('text', '')
        
        # 조건: 점수가 평균보다 높고, 원하는 키워드 중 하나라도 포함하는 경우
        if score >= avg_score * 1.2 and any(keyword in text for keyword in target_keywords):
            high_score_samples.append(sample)

    print(f"\n[🏆 미션 결과]")
    print(f"💡 조건 (점수 >= 평균 * 1.2 & 키워드 포함)을 만족하는 '보석' 콘텐츠는 총 {len(high_score_samples)}개 발견!")
    
    # 발견된 '보석' 콘텐츠의 예시 출력
    print("\n🌟 발견된 보석 콘텐츠 3가지 예시:")
    for i, sample in enumerate(high_score_samples[:3]):
        url = sample.get('url', 'URL 없음')[:40] + '...'
        print(f"  [{i+1}] 출처 URL: {url}")
        # 텍스트가 너무 길지 않도록 처음 50자만 보여줍니다.
        snippet = sample.get('text', '')[:50] + '...'
        print(f"      📜 내용 요약: {snippet}")
    
    
    # 3. LLM 프롬프트 생성 아이디어 (분석 결과를 활용한 창의적 활용)
    
    print("\n======== ✍️ [Step 3] AI 학습 도구 제작: 최고의 프롬프트 만들기 ========")
    print("이 데이터 분석 결과를 가지고, 실제로 LLM에게 질문할 '최고의 프롬프트'를 설계해봅시다.")
    
    # 1. '최적화된 질문' 설계 (분석된 특성을 반영)
    print("🌟 좋은 질문의 조건을 생각해 봅시다:")
    print("1. 출처(URL)를 제한하여 신뢰도를 높인다.")
    print("2. 핵심 키워드(교육/과학 등)를 명시한다.")
    print("3. 원문(text)을 Context로 제공한다.")
    
    # 2. 샘플 데이터의 텍스트를 기반으로 프롬프트 초안 생성
    if high_score_samples:
        sample_text = high_score_samples[0]['text']
        print("\n[✨ 생성된 가상 프롬프트 (LLM 입력창용)]")
        print("----------------------------------------------------------------------")
        print("### 역할 부여 (Role) ###")
        print("당신은 고등학교 과학 선생님입니다. 아래 [Context]에 제공된 정보를 바탕으로, 학생들이 이해하기 쉬운 방식으로 3가지 핵심 질문과 답안을 만들어 주세요.")
        print("\n### 제약 조건 (Constraints) ###")
        print("1. 전문 용어는 쉬운 단어로 반드시 풀어서 설명할 것.")
        print("2. 답안에는 비유나 일상생활 예시를 반드시 포함할 것.")
        print("\n### Context (원문 자료) ###")
        print(f"--- (사용할 원문 내용: '{sample_text[:100]}...') ---")
        print("----------------------------------------------------------------------")
    else:
        print("💡 충분한 샘플이 없어 프롬프트 예시를 보여드릴 수 없어요. 다음에 다시 시도해봅시다!")


# =============================================================================
# 💡 프로그램 시작!
# =============================================================================

# 1. 데이터 로드 함수 호출
sample_data_list = load_data_robustly(
    dataset_id=DATASET_NAME, 
    split='train', 
    sample_count=SAMPLE_COUNT
)

# 2. 탐험 실행
run_data_explorer(sample_data_list)